The aim of the code below is to calculate the base MAE of the predictions that the api data has, so that I can then aim to beat it with my own model.

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sqlalchemy import create_engine
import os

engine = create_engine(os.environ["DATABASE_URL"])

query = """
SELECT
    p.period_from,
    p.period_to,
    nr.reading_id,
    nr.forecast_intensity,
    nr.actual_intensity
FROM national_readings nr
JOIN period_id p
    ON nr.reading_id = p.id
ORDER BY p.period_from;
"""

data = pd.read_sql(query, engine)

data["period_from"] = pd.to_datetime(data["period_from"], utc=True)
# drops readings that have a null intensity
data = data.dropna(subset=["actual_intensity"])
data.head()

data["target"] = data["actual_intensity"]

# Create a copy containing yesterday's readings
yesterday = data[
    ["period_from", "actual_intensity"]
].copy()

# Move yesterday's timestamps forward by one day
# so they line up with today's timestamps
yesterday["period_from"] = (
    yesterday["period_from"] + pd.Timedelta(days=1)
)

# Rename the intensity so we know it came from yesterday
yesterday = yesterday.rename(
    columns={
        "actual_intensity": "yesterday_intensity"
    }
)

# Match today's timestamp with the exact same time yesterday
data = data.merge(
    yesterday,
    on="period_from",
    how="left"
)



In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

baseline = data.dropna(
    subset=["actual_intensity", "yesterday_intensity"]
).copy()

baseline_mae = mean_absolute_error(
    baseline["actual_intensity"],
    baseline["yesterday_intensity"]
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        baseline["actual_intensity"],
        baseline["yesterday_intensity"]
    )
)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)

Baseline MAE: 28.454545454545453
Baseline RMSE: 30.729020929289746
